# Create AAV6-ML plots for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 01/04/2026

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2

## 2. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'Tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for plots
plots_dir = general_dir/'Figures'
os.makedirs(plots_dir, exist_ok=True)

## 3. Define Functions


### 3.1. Helper functions

In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

### 3.2. Script Functions

#### 3.2.1. Function to create Histogram of sample proportion vs input proportion

In [ ]:
# Function to create Histogram of sample proportion vs input proportion

def distribution_histogram (
    table, 
    tissue, 
    extraction, 
    column, 
    replicate, 
    legend = True, 
    title = True, 
    save=False, 
    output_path=None, 
    file_name = None
):
    
    if replicate == "biological":
        legend_name = "Data"
    elif replicate == "technical":
        legend_name = "Sample"
    else:
        raise ValueError("replicate must be 'biological' or 'technical'")
   
    # -----------------------------
    # Prepare input library
    # -----------------------------       
    input_library = dict_library[tissue][extraction].copy()
    input_library[legend_name] = 'input_library'
    
    # -----------------------------
    # Prepare selected sample data
    # -----------------------------    
    df = table[
        (table["replicate"] == replicate) &
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ][['AA_sequence', 'Sample', 'Mouse_ID', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo']].copy()

    # for biological: use Mouse_ID as grouping name
    if replicate == "biological":
        df = df.rename(columns={"Mouse_ID": "Data"})
    
    # concat input library + samples
    df = pd.concat([input_library, df], axis=0, ignore_index=True)

    # keep valid values only
    df = df[df[column].notna()].copy()

    # define x-range
    x_min = table[column].min() 
    x_max = table[column].max()

    bin_edges = np.logspace(np.log10(x_min), np.log10(x_max), 100)
   
    # -----------------------------
    # Styling
    # -----------------------------
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    # -----------------------------
    # Define colors (mouse different blue colours and library red)
    # -----------------------------
    mouse_palette = {
        "f1": "#9ECAE1",   #blau
        "f2": "#6BAED6",
        "f3": "#3182BD",
        "m1": "#D4B9DA",   # violett
        "m2": "#C994C7",   
        "m3": "#88419D"
    }

    input_color = "#BDBDBD"

    # -----------------------------
    # Plot input library first (background)
    # -----------------------------
    input_sub = df[df[legend_name] == "input_library"]
    if not input_sub.empty:
        ax.hist(
            input_sub[column],
            bins=bin_edges,
            alpha=0.8,
            color=input_color,
            label="input_library",
            edgecolor="none",
            zorder=1
        )
        
    # -----------------------------
    # Plot mice on top
    # -----------------------------
    present_data = [x for x in df[legend_name].unique() if x != "input_library"]

    # keep biological mouse order if possible
    ordered_data = [x for x in MOUSE_ID if x in present_data] + [x for x in present_data if x not in MOUSE_ID]

    for data in ordered_data:
        sub = df[df[legend_name] == data]

        color = mouse_palette.get(data, "#4C72B0")  # fallback blue

        ax.hist(
            sub[column],
            bins=bin_edges,
            alpha=0.35,
            color=color,
            label=data,
            edgecolor="none",
            zorder=2
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(x_min, x_max)

    ax.set_xlabel("Variant proportion in sample")
    ax.set_ylabel("Number of AA sequences")

    if title:
        ax.set_title(f"{extraction} {tissue}: distribution of AA-sequence proportions")

    if legend:
        ax.legend(title=legend_name, frameon=False, ncol=2, loc="upper right")

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        fig.savefig(output_path/file_name, dpi=300, bbox_inches="tight")

    plt.show()

#### 3.2.2. Distribution ecdf for Proportion and Log2_enrichment

In [ ]:
def distribution_ecdf(
    table,
    tissue,
    extraction,
    column,
    replicate,
    legend=True,
    title=True,
    save=False,
    output_path=None,
    file_name=None
):
    
    if replicate == "biological":
        legend_name = "Data"
    elif replicate == "technical":
        legend_name = "Sample"
    else:
        raise ValueError("replicate must be 'biological' or 'technical'")

    # -----------------------------
    # Prepare input library
    # -----------------------------
    input_library = dict_library[tissue][extraction].copy()
    input_library[legend_name] = "input_library"

    # -----------------------------
    # Prepare selected sample data
    # -----------------------------
    df_sample = table[
        (table["replicate"] == replicate) &
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ][["AA_sequence", "Sample", "Mouse_ID", f'{column}', "Pseudo"]].copy()

    if replicate == "biological":
        df_sample = df_sample.rename(columns={"Mouse_ID": "Data"})

    if column == 'Proportion':
        df_sample = df_sample[df_sample['Pseudo'] == 0]

    # -----------------------------
    # Concat input + selected samples
    # -----------------------------
    df = pd.concat([input_library, df_sample], axis=0, ignore_index=True)

    # keep valid values only
    df = df[df[column].notna()].copy()

    # define x-range from filtered data
    x_min = df[column].min()
    x_max = df[column].max()

    # -----------------------------
    # Styling
    # -----------------------------
    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    # -----------------------------
    # Define colors
    # -----------------------------
    color_palette = {
        "input_library": "#BDBDBD",  # grey
        "f1": "#9ECAE1",             # blue shades
        "f2": "#6BAED6",
        "f3": "#3182BD",
        "m1": "#D4B9DA",             # violet shades
        "m2": "#C994C7",
        "m3": "#88419D"
    }

    # -----------------------------
    # Define plotting order
    # -----------------------------
    present_samples = [x for x in df[legend_name].dropna().unique() if x != "input_library"]

    preferred_order = ["f1", "f2", "f3", "m1", "m2", "m3"]
    ordered_samples = [x for x in preferred_order if x in present_samples] + \
                      [x for x in present_samples if x not in preferred_order]

    sample_order = ["input_library"] + ordered_samples

    # -----------------------------
    # Plot ECDFs
    # -----------------------------
    for data_name in sample_order:
        sub = df[df[legend_name] == data_name].copy()
        if sub.empty:
            continue

        color = color_palette.get(data_name, "#4C72B0")

        # input in background, full opacity
        if data_name == "input_library":
            alpha_val = 1.0
            lw = 2.2
            z = 1
        else:
            alpha_val = 0.5
            lw = 2.0
            z = 2

        sns.ecdfplot(
            data=sub,
            x=column,
            ax=ax,
            color=color,
            alpha=alpha_val,
            linewidth=lw,
            label=data_name,
            zorder=z
        )

    # -----------------------------
    # Reference lines
    # -----------------------------
    mean_val = df_sample[column].mean()

    # for proportion: only mean line
    # for log2_enrichment: zero line + mean line
    line_handles = []

    if column == "Log2_enrichment":
        line_zero = ax.axvline(
            0,
            linestyle="--",
            linewidth=1.5,
            color="red",
            alpha=0.9,
            label="No enrichment = 0"
        )
        line_handles.append(line_zero)

    line_mean = ax.axvline(
        mean_val,
        linestyle="--",
        linewidth=1.5,
        color="black",
        alpha=0.9,
        label=f"Mean = {mean_val:.2e}" if column == "Proportion" else f"Mean = {mean_val:.2f}"
    )
    line_handles.append(line_mean)

    # -----------------------------
    # Axes
    # -----------------------------
    ax.set_xlim(x_min, x_max)

    if column == "Proportion":
        ax.set_xscale("log")
        ax.set_xlabel("Variant proportion")
        ylab = "Cumulative fraction of AA sequences"
        title_text = f"{extraction} {tissue}\nCumulative distribution of AA-sequence proportions"
    else:
        ax.set_xlabel(f"Variant {column}")
        ylab = "Cumulative fraction of AA sequences"
        title_text = f"{extraction} {tissue}\nCumulative distribution of {column}"

    ax.set_ylabel(ylab)

    if title:
        ax.set_title(title_text)

    # -----------------------------
    # Legend
    # -----------------------------
    if legend:
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))  # remove duplicates, preserve last occurrence
        ax.legend(
            unique.values(),
            unique.keys(),
            title= f'{extraction} {tissue}',
            loc="lower right" if column == "Proportion" else "upper right",
            frameon=False,
            ncol=2
        )

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        if file_name is None:
            file_name = f"ECDF_{column}_{extraction}_{tissue}_{replicate}.png"
        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    plt.show()

#### 3.2.3. Venn2 Diagram of top proportion between sample and input

In [ ]:
def venn2_ntop_AA (table, tissue, extraction, n_top, title = True, save = False, output_path = None):

    # select samples I want to compare
    sample = ((table [
        (table['Tissue'] == tissue) &
        (table['Extraction_type'] == extraction)
    ][["AA_sequence", 'Proportion']].sort_values('Proportion', ascending = False)).head(n_top)).copy()
    
    
    #select
    input_library = ((dict_library[tissue][extraction][["AA_sequence", 'Proportion']].sort_values('Proportion', ascending = False)).head(n_top)).copy()
    
        
    plt.figure(figsize=(7,4))
    venn2([set(input_library["AA_sequence"]), set(sample["AA_sequence"])], set_labels=("Input library", f"{tissue} {extraction} pooled"))

    if title:
        plt.title(f"{extraction} {tissue}\nOverlap of top {n_top} proportion AA variants")
    
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)
        plt.savefig(output_path/f'Venn2_{extraction}_{tissue}_proportion_top_{n_top}', dpi=300, bbox_inches="tight")

    
    plt.show()

#### KDE distribution for Chapter 3

In [ ]:
def distribution_kde(table, tissue, extraction, column, save=False, output_path=None):

    input_library = dict_library[tissue][extraction].copy()

    df = table[
        (table["Tissue"] == tissue) &
        (table["Extraction_type"] == extraction)
    ][['AA_sequence', 'Mouse_ID', 'Proportion', 'Cum_prop', 'Log2_enrichment', 'Pseudo']].copy()

    df = df.rename(columns={'Mouse_ID': 'Data'})

    # 👉 concat statt merge
    df = pd.concat([input_library, df], axis=0, ignore_index=True)

    # remove NaN + zeros (wichtig für log!)
    df = df[df[column].notna()].copy()
    df = df[df[column] > 0].copy()

    # 👉 log-transform für KDE
    df["log_value"] = np.log10(df[column])

    sns.set_style("white")
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10
    })

    fig, ax = plt.subplots(figsize=(7.5, 5.2))

    # KDE plot
    sns.kdeplot(
        data=df,
        x="log_value",
        hue="Data",
        common_norm=False,   # wichtig für Vergleich!
        fill=True,
        alpha=0.3,
        linewidth=1.5,
        ax=ax
    )

    # Achsen zurück in log-space beschriften
    xticks = np.arange(np.floor(df["log_value"].min()), np.ceil(df["log_value"].max()))
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"$10^{{{int(x)}}}$" for x in xticks])

    ax.set_xlabel("Variant proportion")
    ax.set_ylabel("Density")
    ax.set_title(f"{extraction} {tissue}: distribution of AA-sequence proportions (KDE)")

    ax.legend(title="Data", frameon=False)

    sns.despine()
    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    plt.show()

## 4. Script
### 4.1. Load CSV Files

In [ ]:
%%time
#load long table
df_long_table = read_csv_file(tables_dir / 'summary', "df_long_combined_bio_tech")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_pooled_table")

In [ ]:
print ('Long Table')
display (df_long_table)

print ('Pooled Table')
display (df_pooled)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long_table, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
# Sort different file names in extraction and biological or technical replicates
DICT_NAMES = sort_file_names (SAMPLE)

In [ ]:
DICT_NAMES

### 4.3. Load tissue/ext specific librarys

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue, ext in product(TISSUE, EXT):
    df = read_csv_file(tables_dir/tissue, f'df_input_library_for_{ext}_{tissue}')
    dict_library.setdefault(tissue,{})[ext] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

### 4.4. Comparison between input and samples regarding proportion

#### 4.4.1. Histogram with am proportion with Hue = Sample


In [ ]:
%%time
for ext, tissue in product (EXT, TISSUE):
    distribution_histogram (df_long_table, tissue, ext, 'Proportion', 'biological', legend = True, title = True)


#### 4.4.2. ECDF with samples and input

In [ ]:
for ext, tissue in product (EXT, TISSUE):
    distribution_ecdf(
        table=df_long_table,
        tissue=tissue,
        extraction=ext,
        column="Proportion",
        replicate="biological",
        save = True,
        output_path = plots_dir/'1_proportion'/'2_ECDF_proportion'     
    )

#### 4.4.3. Venn2 Input vs mean proportion in sample

In [ ]:
for ext, tissue in product (EXT, TISSUE):
    venn2_ntop_AA (df_pooled, tissue, ext, 10000, save = True, title=True, output_path = plots_dir/'1_proportion'/'3_Venn2')

In [ ]:
number = [10, 100, 1000, 10000]
for ext, tissue, n in product (EXT, TISSUE, number):
    venn2_ntop_AA (df_pooled, tissue, ext, n)

### 4.5. ECDF Log2 distribution between samples

In [ ]:
%%time
for tissues, exts in product (TISSUE, EXT):
    distribution_ecdf (table = df_long_table, tissue = tissues, extraction = exts, column = 'Log2_enrichment', replicate = 'biological', save = True, output_path = plots_dir/'2_enrichment'/'1_ECDF');

### 4.6. Comparison functional selection between RNA and DNA samples

#### 4.6.1. Violin plot mean between biological replicate

#### 4.6.2. KDE plot for liver and heart (DNA and RNA in one plot, x = enrichment, y = n variant)

#### 4.6.3. Venn2 plot with top 10000 enrichment
    - to show already the selection difference between biological step and tissue

#### 4.6.4. Diagramm of Venn2 plot overlap between top 1 and top 10^6

## 99. Suplement Figures

### 99.1. Correlation between raw input Library and tissue libraries

In [ ]:
input_library

In [ ]:
tissue = 'liver'
extraction = 'cDNA'

tissue_specific = dict_library[tissue][extraction].sort_values('AA_sequence', ascending = True).copy()
input_library = df_raw_input.sort_values('AA_sequence', ascending = True).copy()
input_library['Proportion'] = input_library['Count']/sum(input_library['Count'])

# merge über AA_sequence
merged = tissue_specific.merge(
    input_library[['AA_sequence', 'Proportion']],
    on='AA_sequence',
    how='inner',
    suffixes=('_tissue', '_input')
)

merged = merged.dropna(subset=['Proportion_tissue', 'Proportion_input'])

# Pearson correlation
pearson_corr = stats.pearsonr(
    merged['Proportion_tissue'],
    merged['Proportion_input']
)
pearson_corr

sns.scatterplot(data = merged, x = 'Proportion_input', y = 'Proportion_input');
sns.axline(slope = float(pearson_corr[0]), color = 'b')



In [ ]:
df_wide = (
    df_pooled.pivot_table(
        index=["AA_sequence", "Tissue"],
        columns="Extraction_type",
        values="Log2_enrichment",
        aggfunc="first"
    )
)

df_wide = df_wide.rename(columns={
    "gDNA": "Log2_enrichment_gDNA",
    "cDNA": "Log2_enrichment_cDNA"
}).reset_index()

df_wide

In [ ]:
sns.jointplot(
    data=df_wide,
    x="Log2_enrichment_gDNA",
    y="Log2_enrichment_cDNA",
    hue="Tissue",
    kind="scatter",
    marginal_kws=dict(common_norm=False),
    alpha=0.4
)